# Backtest — Silver Bullet & OTE

Runs the automatable rules against your own price data.

**Nothing installs on your computer.** This runs on Google's machines.

---

## How to run it

1. **Runtime → Run all**
2. Step 3 will ask you to pick a file — choose your 1-minute CSV
3. Read Step 4 **before** looking at any results
4. Copy the output back into the chat

## What you need first

A 1-minute CSV with a timezone on the timestamps. **2 years** if you want
results that mean anything — the trading windows are one hour a day, so a
short file produces a confident wrong answer rather than a cautious one.

## Step 1 — Get the code

In [ ]:
import os, shutil, subprocess

REPO = "https://github.com/dboy140/Dboytrades.git"
BRANCH = "claude/ict-nbbtrader-trading-system-43hipg"

if os.path.isdir("/content/Dboytrades"):
    shutil.rmtree("/content/Dboytrades")

r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO, "/content/Dboytrades"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit(f"Could not download the code:\n{r.stderr}")

os.chdir("/content/Dboytrades")
os.makedirs("data/bars", exist_ok=True)
print("Step 1 done.")

## Step 2 — Self-test

Runs the test suite. Needs no internet and proves the engine is intact before
you spend time on results.

In [ ]:
!pip install -q pytest
!python -m pytest -q 2>&1 | tail -3

## Step 3 — Upload your CSV

A file picker will appear. Large files take a few minutes.

In [ ]:
import glob, shutil
from google.colab import files

uploaded = files.upload()
assert uploaded, "No file chosen."

name = list(uploaded)[0]
dest = f"data/bars/{name}"
shutil.move(name, dest)

rows = sum(1 for _ in open(dest)) - 1
print(f"\n{dest}")
print(f"{rows:,} rows")
if rows < 100_000:
    print("\nNOTE: under ~100k rows is less than a few months of 1m data.")
    print("It will run, but treat any result as a smoke test, not evidence.")

## Step 4 — Validate the data ⚠️ read this output

This is the step that stops you wasting a day. It checks the timezone against
the data itself, and reports how many bars actually fall inside each trading
window.

**If session coverage is 0 for the windows you care about, the backtest cannot
test those rules — however many rows the file has.**

In [ ]:
!python -m bot.inspect_data "$dest" 2>&1 | tail -40

### If Step 4 reported a timezone drift

Uncomment the line below, replacing `-3` with the **negative** of the drift it
reported, then re-run this cell and Step 4.

In [ ]:
# !python -m bot.inspect_data "$dest" --restamp -3
# dest = dest.rsplit('.', 1)[0] + '.fixed.csv'
# print('now using', dest)

## Step 5 — Run it

Set the instrument to match your file. EURUSD/GBPUSD test the OTE rules and
NBB's windows; NAS100 tests the Silver Bullet windows.

In [ ]:
import subprocess, sys

INSTRUMENT = "EURUSD"   # EURUSD | GBPUSD | XAUUSD | NAS100

# FX and gold run the OTE rules; the index runs the Silver Bullet windows,
# which additionally need a directional bias.
if INSTRUMENT in ("EURUSD", "GBPUSD", "XAUUSD"):
    cmd = [sys.executable, "-m", "bot.run_backtest", dest,
           "--instrument", INSTRUMENT, "--setup", "ote"]
else:
    cmd = [sys.executable, "-m", "bot.run_backtest", dest,
           "--instrument", INSTRUMENT, "--setup", "silver_bullet",
           "--bias", "auto"]

print(" ".join(cmd), "\n")
print(subprocess.run(cmd, capture_output=True, text=True).stdout)

---

## Done — copy back into the chat

1. The **Step 4** validation output (especially session coverage)
2. The **Step 5** results

### Reading the numbers

| Field | What it tells you |
| --- | --- |
| `expectancy_r` | Average R per trade. Below 0 means the rule lost money. |
| `max_consecutive_losses` | Sizing must survive this at 1% per trade. |
| `avg_mae_r_winners` | How close winners came to the stop. Consistently small = stops wider than needed. |
| `avg_mfe_r_losers` | How far losers ran your way first. Consistently large = exits too late. |
| `per rule` | The important one. A rule below breakeven over 20+ trades is a demotion candidate. |

**Under 20 trades per rule, nothing is conclusive.** The tool says so itself.